In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
import xmlrpc.client
import threading
import time
import datetime

# СЕРВЕР
main_server = xmlrpc.client.ServerProxy(
    "http://localhost:8008",
    allow_none=True
)

# СТАТ
stats_client = xmlrpc.client.ServerProxy(
    "http://localhost:8028",
    allow_none=True
)

N = 5  # запросы огр
T = 5  # секунды огр

request_times = []


def wait_for_limit():
    limit_logged = False

    while True:
        now = time.monotonic()

        # Удаляем время запросов, которые уже вышли из интервала.
        while request_times and now - request_times[0] >= T:
            request_times.pop(0)

        # Если лимит не исчерпан — разрешаем запрос.
        if len(request_times) < N:
            request_times.append(now)
            return

        # Один раз записываем превышение лимита.
        if not limit_logged:
            try:
                stats_client.add_log("too_many_requests")
            except Exception as error:
                print("Не удалось записать статистику:", error)

            limit_logged = True

        # Ждём освобождения места для следующего запроса.
        delay = request_times[0] + T - time.monotonic()
        time.sleep(max(0, delay))


class Proxy:
    def _dispatch(self, method, params):
        # Если клиент просит журнал — получаем его из статистики.
        if method == "get_log":
            try:
                return stats_client.get_log(*params)
            except Exception as error:
                return f"Stats server unavailable: {error}"

        # Ждём, если превышен лимит запросов.
        wait_for_limit()

        # Запоминаем дату и время начала вызова.
        timestamp = datetime.datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        # Запускаем отсчёт длительности.
        start = time.monotonic()

        try:
            # Вызываем функцию основного сервера.
            result = getattr(main_server, method)(*params)
        finally:
            # Считаем, сколько секунд занял вызов.
            duration = time.monotonic() - start

            # Записываем событие вместе с его длительностью.
            try:
                stats_client.add_log(method, timestamp, duration)
            except Exception as error:
                print("Не удалось записать статистику:", error)

        return result


proxy_server = SimpleXMLRPCServer(
    ("127.0.0.1", 8009),
    allow_none=True,
    logRequests=False
)

proxy_server.register_instance(Proxy())

proxy_server.serve_forever()

print("Прокси запущен на порту 8009")